# Setup

**Chapter 4. Reasoning & Planning — Code Repair Agent**

Run cells from top to bottom. The default run uses **Ollama + `qwen3:4b`** and the adaptive graph.
The ReAct, fixed-plan, and forced-replanning runs are optional switches below.

| Section | Focus |
|---|---|
| 1. Task & Tools | A buggy function, fixed acceptance tests, and three tools |
| 2. ReAct | A tool-using agent without an explicit plan in State |
| 3. Plan-and-Execute | A structured plan and one-step execution |
| 4. Replanning | Replace the remaining plan using new evidence |
| 5. Verification & Reflection | External checks, concise feedback, and bounded execution |

**Before running:** start Ollama and download `qwen3:4b`. No API key is required. [1]

```bash
ollama pull qwen3:4b
# Run `ollama serve` only if the Ollama service is not already running.
```

<details>
<summary>Note) Dependencies and execution scope</summary><br>

The notebook uses LangChain 1.x, LangGraph 1.x, langchain-ollama 1.x, Pydantic 2.x,
and Python 3.11+. Use your existing notebook kernel or a separate virtual environment.
The commented install command below does not run automatically.

Only a temporary `math_utils.py` is modified. The tools do not accept arbitrary file paths
or shell commands. Generated functions are restricted to a small Python subset and checked
in a subprocess with a timeout. **These measures are not a production security sandbox.**
Do not extend the runner to arbitrary repositories or untrusted code without proper isolation.

All explanations, prompts, and comments in this notebook are in English.
The organization follows your uploaded architecture notebook; the implementation below is new.

</details>

In [1]:
# Optional: install in this notebook's environment, then restart the kernel.
# %pip install "langchain>=1,<2" "langgraph>=1,<2" "langchain-ollama>=1,<2" "pydantic>=2,<3"

import ast
import hashlib
import json
import subprocess
import sys
from importlib.metadata import version
from operator import add
from pathlib import Path
from pprint import pprint
from tempfile import TemporaryDirectory
from textwrap import dedent
from typing import Annotated, Literal
from typing_extensions import TypedDict

from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

In [2]:
# 1. Settings
seed = 42
MAX_MODEL_CALLS = 6          # Per agent invocation, not per complete graph run
MAX_STEPS = 6               # Outer Executor invocations
MAX_REPLANS = 2             # Replacements of the remaining plan
TEST_TIMEOUT = 5            # Seconds per test subprocess

RUN_REACT_DEMO = False
RUN_FIXED_PLAN_DEMO = False
RUN_ADAPTIVE_DEMO = True
RUN_FORCED_REPLAN_DEMO = False


# 2. Chat model
chat = ChatOllama(
    model="qwen3:4b",
    temperature=0,
    reasoning=False,
    seed=seed,
    num_ctx=8192,
    num_predict=2048,
    client_kwargs={"timeout": 120.0},
)

# Model reasoning mode is independent of our explicit plan in State.
for package in ["langchain", "langgraph", "langchain-ollama", "pydantic"]:
    print(f"{package}: {version(package)}")

langchain: 1.4.5
langgraph: 1.2.15
langchain-ollama: 1.1.0
pydantic: 2.13.5


# Terminologies

| Term | Meaning in this notebook |
|---|---|
| Reasoning | Use current information to choose an action or form a hypothesis. |
| Planning | Define actionable tasks, their order, and completion criteria. |
| Observation | An actual tool result, not an agent's claim. |
| ReAct-style execution | Alternate action selection and tool observations. |
| Plan-and-Execute | Separate planning from execution of individual tasks. |
| Replanning | Revise the remaining tasks when observations require a new approach. |
| Verification | Check the current code against fixed acceptance tests. |
| Reflection | Produce a short, evidence-based adjustment for the next attempt. |

**Graph = control structure. Plan = data.** The graph remains fixed while `state["plan"]` changes.
This example executes plan items sequentially; list order represents dependencies.
It does not implement a dependency DAG, a multi-agent team, or model-weight training. [2–4]

# 1. Task & Tools

## 1.1. Goal and acceptance criteria

Repair `average(values)` without changing its name or argument.
For this exercise, `values` is a list of finite numbers: return `0.0` for an empty list,
and its arithmetic mean otherwise. No new packages or files are needed.

| Category | Example | Expected |
|---|---|---|
| Empty input | `[]` | `0.0` |
| Fractional result | `[2, 3]` | `2.5` |
| Regression | `[2, 4]` | `3.0` |
| Single value | `[-4]` | `-4.0` |
| Negative values | `[-2, -3]` | `-2.5` |
| Decimal values | `[0.1, 0.2, 0.3]` | Approximately `0.2` |

The test cases belong to the controller. The agent cannot edit them through its tools.
A successful file write does **not** mean the repair succeeded.

In [3]:
# 1. Task
GOAL = (
    "Repair average(values) in math_utils.py. Keep the function name and argument. "
    "For an empty list, return 0.0. Otherwise return the arithmetic mean. "
    "Handle fractional results, negative numbers, and decimal inputs correctly."
)

BUGGY_SOURCE = "def average(values):\n    return sum(values) // len(values)\n"

# A manually authored partial fix for the optional replanning demonstration.
PARTIAL_SOURCE = (
    "def average(values):\n"
    "    if not values:\n"
    "        return 0.0\n"
    "    return sum(values) // len(values)\n"
)

TESTS = [
    {"name": "empty", "values": [], "expected": 0.0},
    {"name": "fractional", "values": [2, 3], "expected": 2.5},
    {"name": "regression", "values": [2, 4], "expected": 3.0},
    {"name": "single", "values": [-4], "expected": -4.0},
    {"name": "negative", "values": [-2, -3], "expected": -2.5},
    {"name": "decimal", "values": [0.1, 0.2, 0.3], "expected": 0.2},
]


# 2. Workspace: no existing project files are overwritten.
workspace = TemporaryDirectory(prefix="agent_ch4_")
source_path = Path(workspace.name) / "math_utils.py"


def reset_workspace(source: str = BUGGY_SOURCE):
    source_path.write_text(source, encoding="utf-8")


def revision(source: str) -> str:
    return hashlib.sha256(source.encode()).hexdigest()[:16]


reset_workspace()
print(source_path.read_text(encoding="utf-8"))

def average(values):
    return sum(values) // len(values)



## 1.2. Test runner

`verify_source()` executes real Python in a separate process and returns a structured report.
A source hash associates the report with the exact code tested. [5]

| Report status | Meaning |
|---|---|
| `passed` | All fixed checks passed for this code revision. |
| `failed` | A behavior check or source-validation check failed. |
| `blocked` | The test subprocess could not complete reliably. |

<details>
<summary>Note) Small helper code, not the planning algorithm</summary><br>

The next two cells are execution infrastructure. Read the three tools in 1.3 first if the
AST validator distracts from the agent concepts. AST means Python's abstract syntax tree.

The accepted source is one undecorated `def average(values):` function. It may use
`if`, `return`, arithmetic, comparisons, and `sum`, `len`, or `float` calls.
Imports, attributes, loops, assignments, type annotations, and arbitrary function calls
are intentionally excluded. This keeps the exercise narrow; it is not a general coding agent.
The runner does not prove that all possible inputs work.

</details>

In [4]:
# 1. Validate the small source-code subset before writing or executing it.
ALLOWED_AST = (
    ast.Module, ast.FunctionDef, ast.arguments, ast.arg, ast.Return,
    ast.If, ast.IfExp, ast.Compare, ast.UnaryOp, ast.Not, ast.USub, ast.UAdd,
    ast.BinOp, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv,
    ast.Constant, ast.Name, ast.Load, ast.Call, ast.Expr,
    ast.Eq, ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE,
)


def validate_source(source: str):
    if len(source) > 3000:
        raise ValueError("The function must be at most 3000 characters.")
    tree = ast.parse(source)
    if len(tree.body) != 1 or not isinstance(tree.body[0], ast.FunctionDef):
        raise ValueError("Provide exactly one function: def average(values):")

    function = tree.body[0]
    args = function.args
    if (function.name != "average" or [a.arg for a in args.args] != ["values"]
            or args.posonlyargs or args.kwonlyargs or args.defaults
            or args.vararg or args.kwarg or function.decorator_list
            or function.returns or any(a.annotation for a in args.args)):
        raise ValueError("Keep the exact unannotated signature: def average(values):")

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node is not function:
            raise ValueError("Nested functions are not supported.")
        if not isinstance(node, ALLOWED_AST):
            raise ValueError(f"Unsupported syntax: {type(node).__name__}")
        if isinstance(node, ast.Name) and node.id not in {"values", "sum", "len", "float"}:
            raise ValueError(f"Unsupported name: {node.id}")
        if isinstance(node, ast.Call):
            if (not isinstance(node.func, ast.Name)
                    or node.func.id not in {"sum", "len", "float"} or node.keywords):
                raise ValueError("Only sum(), len(), and float() calls are allowed.")
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)) and abs(node.value) > 1_000_000:
                raise ValueError("Numeric constants are too large for this exercise.")
    compile(tree, "math_utils.py", "exec")

In [5]:
# 2. The runner and test cases are not editable through agent tools.
RUNNER = r"""
import json, math, sys
payload = json.load(sys.stdin)
namespace = {"__builtins__": {"sum": sum, "len": len, "float": float}}
exec(compile(payload["source"], "math_utils.py", "exec"), namespace)
rows = []
for case in payload["tests"]:
    row = {"name": case["name"], "expected": case["expected"]}
    try:
        actual = namespace["average"](list(case["values"]))
        passed = type(actual) in (int, float) and math.isclose(
            actual, case["expected"], rel_tol=1e-9, abs_tol=1e-9)
        if not case["values"]:
            passed = passed and type(actual) is float
        row.update(actual=repr(actual), passed=bool(passed))
    except Exception as error:
        row.update(passed=False, error=f"{type(error).__name__}: {error}")
    rows.append(row)
print(json.dumps(rows))
"""


def verify_source(source: str) -> dict:
    report = {"revision": revision(source), "passed": 0, "total": len(TESTS), "cases": []}
    try:
        validate_source(source)
    except (ValueError, SyntaxError) as error:
        return {**report, "status": "failed", "error": str(error)}

    try:
        process = subprocess.run(
            [sys.executable, "-I", "-c", RUNNER],
            input=json.dumps({"source": source, "tests": TESTS}),
            capture_output=True, text=True, timeout=TEST_TIMEOUT,
        )
        if process.returncode != 0:
            return {**report, "status": "blocked", "error": process.stderr[-1000:]}
        rows = json.loads(process.stdout)
    except (OSError, subprocess.TimeoutExpired, json.JSONDecodeError) as error:
        return {**report, "status": "blocked", "error": f"{type(error).__name__}: {error}"}

    passed = sum(row["passed"] for row in rows)
    return {**report, "cases": rows, "passed": passed,
            "status": "passed" if passed == len(TESTS) else "failed"}


baseline_report = verify_source(BUGGY_SOURCE)
pprint(baseline_report)

{'cases': [{'error': 'ZeroDivisionError: division by zero',
            'expected': 0.0,
            'name': 'empty',
            'passed': False},
           {'actual': '2',
            'expected': 2.5,
            'name': 'fractional',
            'passed': False},
           {'actual': '3',
            'expected': 3.0,
            'name': 'regression',
            'passed': True},
           {'actual': '-4', 'expected': -4.0, 'name': 'single', 'passed': True},
           {'actual': '-3',
            'expected': -2.5,
            'name': 'negative',
            'passed': False},
           {'actual': '0.0',
            'expected': 0.2,
            'name': 'decimal',
            'passed': False}],
 'passed': 2,
 'revision': '65859fe329efbb61',
 'status': 'failed',
 'total': 6}


## 1.3. Tools

| Tool | Operation |
|---|---|
| `read_file()` | Read the current source and its revision. |
| `apply_patch(source)` | Replace the complete tiny function after source validation. |
| `run_tests()` | Execute the fixed tests on the current source. |

`apply_patch` accepts the complete function, not a Git diff.
These tools operate only on the temporary file created above.

In [6]:
@tool
def read_file() -> dict:
    """Read math_utils.py and its revision. No path argument is needed."""
    source = source_path.read_text(encoding="utf-8")
    return {"source": source, "revision": revision(source)}


@tool
def apply_patch(source: str) -> dict:
    """Replace math_utils.py with one complete def average(values): function.

    Use plain Python without Markdown, imports, annotations, or extra functions.
    Only if/return, arithmetic, comparisons, sum(), len(), and float() are supported.
    """
    try:
        validate_source(source)
    except (ValueError, SyntaxError) as error:
        return {"ok": False, "error": str(error)}
    previous = source_path.read_text(encoding="utf-8")
    source_path.write_text(source, encoding="utf-8")
    return {"ok": True, "changed": source != previous, "revision": revision(source)}


@tool
def run_tests() -> dict:
    """Run the fixed acceptance tests against the current math_utils.py."""
    return verify_source(source_path.read_text(encoding="utf-8"))

# 2. ReAct

The executor below uses a bounded model–tool loop. It can read observations and select another
action without storing an explicit multi-step plan. This is a **ReAct-style baseline**, not a
reproduction of the original paper's prompting or benchmarks. [2]

We record actual tool results, not hidden reasoning. `reasoning=False` does not disable tool use
or the explicit planning that we add later.

In [7]:
# 1. Agent: reuse the same executor for full-task and single-step requests.
executor = create_agent(
    model=chat,
    tools=[read_file, apply_patch, run_tests],
    system_prompt=(
        "You are a code-repair executor. Use the tools; never fabricate their results. "
        "Complete only the requested task, then return a brief English summary. "
        "Before a patch, read the current source. Submit the entire function, not a diff. "
        "Keep def average(values): without annotations. Use simple if/return and "
        "arithmetic with sum(), len(), or float(). Do not use imports or assignments. "
        "A patch success is not a test pass. Report failure or missing evidence honestly."
    ),
    middleware=[ModelCallLimitMiddleware(run_limit=MAX_MODEL_CALLS, exit_behavior="end")],
)


def agent_config() -> dict:
    return {"recursion_limit": 2 * MAX_MODEL_CALLS + 8}


def tool_observations(messages) -> list[dict]:
    observations = []
    for message in messages:
        if isinstance(message, ToolMessage):
            try:
                result = json.loads(message.content)
            except (TypeError, json.JSONDecodeError):
                result = {"text": str(message.content)}
            observations.append({"tool": message.name, "result": result})
    return observations

In [8]:
# 2. Optional baseline run: independent of the planning graph.
if RUN_REACT_DEMO:
    reset_workspace()
    react_result = executor.invoke(
        {"messages": [HumanMessage(content=GOAL + " Run the tests before and after editing.")]},
        config=agent_config(),
    )
    pprint(tool_observations(react_result["messages"]))

    # The controller verifies again; it does not trust the final agent message.
    react_report = verify_source(source_path.read_text(encoding="utf-8"))
    print("Verified status:", react_report["status"])
    print("Tests:", react_report["passed"], "/", react_report["total"])
else:
    print("ReAct demo skipped. Set RUN_REACT_DEMO = True to run it.")

ReAct demo skipped. Set RUN_REACT_DEMO = True to run it.


# 3. Plan-and-Execute

## 3.1. Structured plan

The planner returns a short list of steps. The executor receives **one step at a time** and may
make several tool calls inside that step. Both roles reuse `chat`; separate roles do not require
separate model weights. `ToolStrategy` returns the validated model in `structured_response`. [3, 6]

| Field | Example |
|---|---|
| `kind` | `inspect`, `repair`, or `test` |
| `task` | `Handle empty input without changing the function signature.` |
| `done_when` | `The requested edit is applied and the write result is recorded.` |

`done_when` communicates intent; it is **not an executable guarantee**. The controller checks
actual tool observations for basic step completion and uses external tests for task success.

In [9]:
# 1. Output schema
class Step(BaseModel):
    kind: Literal["inspect", "repair", "test"]
    task: str = Field(description="One concrete action for the executor.")
    done_when: str = Field(description="A short, observable completion criterion.")


class Plan(BaseModel):
    steps: list[Step] = Field(min_length=1, max_length=4)


# 2. Planner: structured output, no workspace-writing tools.
planner = create_agent(
    model=chat,
    tools=[],
    system_prompt=(
        "Plan a small code repair. Return 1 to 4 remaining steps using the Plan schema. "
        "Use kind inspect, repair, or test. Steps execute sequentially. "
        "The current source and an actual test report are supplied. Do not repeat "
        "completed work unnecessarily. Include a repair for failing behavior and a "
        "test step after edits. Do not put full source code in the plan. "
        "When revising, use the new evidence and preserve completed work."
    ),
    response_format=ToolStrategy(Plan),
    middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")],
)


def request_structured(agent, prompt: str):
    result = agent.invoke(
        {"messages": [HumanMessage(content=prompt)]}, config=agent_config(),
    )
    if "structured_response" not in result:
        raise RuntimeError("The agent ended without valid structured output. Check its call budget and model.")
    return result["structured_response"]

## 3.2. State

| Field | Update rule | Purpose |
|---|---|---|
| `code`, `test_report` | Replace | Current source and independently checked evidence |
| `plan` | Replace | Remaining steps, not a graph definition |
| `past_steps` | Append | Attempts, observed tool results, and basic completion flags |
| `plan_history` | Append | Initial and revised plans |
| `reflection` | Replace | Latest short adjustment; not proof of correctness |
| `steps_used`, `replans` | Explicit increment | Outer execution budgets |
| `unchanged_repairs` | Increment/reset | Detect repeated repair attempts without code changes |
| `status`, `error` | Replace | Running, succeeded, incomplete, or blocked |

A failed test run can still be a **completed test operation**. It is not a successful repair.
`past_steps` records attempts, including unsuccessful ones; it does not certify task completion.

In [10]:
# 1. State
class State(TypedDict):
    goal: str
    code: str
    plan: list[dict]
    past_steps: Annotated[list[dict], add]
    plan_history: Annotated[list[list[dict]], add]
    reflection: str
    test_report: dict
    steps_used: int
    replans: int
    unchanged_repairs: int
    status: Literal["running", "succeeded", "incomplete", "blocked"]
    error: str


def initial_state() -> State:
    return State(
        goal=GOAL, code=source_path.read_text(encoding="utf-8"),
        plan=[], past_steps=[], plan_history=[], reflection="", test_report={},
        steps_used=0, replans=0, unchanged_repairs=0, status="running", error="",
    )


# 2. Shared context: bounded and explicit, rather than an unlimited chat transcript.
def task_context(state: State) -> str:
    recent = [
        {"step": item["step"], "completed": item["completed"], "changed": item["changed"]}
        for item in state["past_steps"][-3:]
    ]
    return json.dumps({
        "goal": state["goal"], "source": state["code"],
        "test_report": state["test_report"], "remaining_plan": state["plan"],
        "recent_attempts": recent, "reflection": state["reflection"],
    }, indent=2)


def blocked(error: Exception) -> dict:
    # Infrastructure or model-output errors are not application test failures.
    return {"status": "blocked", "error": f"{type(error).__name__}: {error}"}

In [11]:
# 3. Nodes: observe before planning, and always verify after execution.
def inspect_code(state: State):
    source = source_path.read_text(encoding="utf-8")
    report = verify_source(source)
    return {"code": source, "test_report": report}


def make_plan(state: State):
    try:
        result = request_structured(planner, "Create the initial plan.\n" + task_context(state))
        steps = [step.model_dump() for step in result.steps]
        return {"plan": steps, "plan_history": [steps]}
    except Exception as error:
        return blocked(error)


def verify_code(state: State):
    # The same ordinary Python function is used for baseline and post-edit checks.
    return inspect_code(state)


def operation_completed(kind: str, observations: list[dict]) -> bool:
    expected = {"inspect": "read_file", "repair": "apply_patch", "test": "run_tests"}[kind]
    for item in observations:
        result = item["result"]
        if item["tool"] != expected or not isinstance(result, dict):
            continue
        if kind == "inspect" and "source" in result:
            return True
        if kind == "repair" and result.get("ok") is True:
            return True
        if kind == "test" and result.get("status") in {"passed", "failed"}:
            return True
    return False

In [12]:
# 4. Execute one item. Only observed operations can consume that item.
def execute_step(state: State):
    step = state["plan"][0]
    before = source_path.read_text(encoding="utf-8")
    try:
        prompt = (
            "Execute only the current step. Use its matching tool; do not merely describe it.\n"
            f"Current step: {json.dumps(step)}\n" + task_context(state)
        )
        result = executor.invoke(
            {"messages": [HumanMessage(content=prompt)]}, config=agent_config(),
        )
        observations = tool_observations(result["messages"])
    except Exception as error:
        return {**blocked(error), "steps_used": state["steps_used"] + 1}

    after = source_path.read_text(encoding="utf-8")
    changed = revision(before) != revision(after)
    completed = operation_completed(step["kind"], observations)
    unchanged = state["unchanged_repairs"]
    if step["kind"] == "repair":
        unchanged = 0 if changed else unchanged + 1

    entry = {
        "step": step, "completed": completed, "changed": changed,
        "revision": revision(after), "observations": observations,
        "assistant_messages": sum(isinstance(m, AIMessage) for m in result["messages"]),
    }
    return {
        "code": after, "plan": state["plan"][1:] if completed else state["plan"],
        "past_steps": [entry], "steps_used": state["steps_used"] + 1,
        "unchanged_repairs": unchanged,
        "test_report": {},  # Previous evidence must not certify a new revision.
    }

In [13]:
# 5. Completion and budget checks: the model cannot override these functions.
def verified(state: State) -> bool:
    report = state["test_report"]
    current = source_path.read_text(encoding="utf-8")
    return (
        report.get("status") == "passed"
        and report.get("passed") == len(TESTS)
        and report.get("revision") == revision(state["code"]) == revision(current)
    )


def stop_reason(state: State) -> str:
    if state["status"] == "blocked":
        return state["error"] or "Agent execution was blocked."
    if state["test_report"].get("status") == "blocked":
        return state["test_report"].get("error") or "Test execution was blocked."
    if state["unchanged_repairs"] >= 2:
        return "Two repair attempts made no source change."
    if state["steps_used"] >= MAX_STEPS:
        return "The outer execution-step budget was exhausted."
    return ""


def finish(state: State):
    if not verified(state):
        return {"status": "incomplete", "error": "No passing report for the current revision."}
    return {"status": "succeeded", "error": ""}


def stop(state: State):
    is_blocked = state["status"] == "blocked" or state["test_report"].get("status") == "blocked"
    return {
        "status": "blocked" if is_blocked else "incomplete",
        "error": stop_reason(state) or "Acceptance criteria are unmet; the plan or replanning budget is exhausted.",
    }


# 6. Routing for the fixed-plan baseline.
def after_inspect(state: State):
    return "finish" if verified(state) else "stop" if stop_reason(state) else "planner"


def after_plan(state: State):
    return "stop" if stop_reason(state) or not state["plan"] else "execute"


def fixed_after_verify(state: State):
    if verified(state):
        return "finish"
    if stop_reason(state):
        return "stop"
    if state["plan"] and state["past_steps"][-1]["completed"]:
        return "execute"
    return "stop"

## 3.3. Fixed-plan graph

```text
START -> inspect -> planner -> execute -> verify
                                  ^       |
                                  |       +-> finish -> END   (current tests pass)
                                  +-------+                  (steps remain)
                                          +-> stop -> END     (blocked or exhausted)
```

This version does not replace a failed plan. It can therefore stop as `incomplete` even though
all planned operations ran. It may finish early when current acceptance tests already pass.

In [14]:
# 1. Graph
builder = StateGraph(State)
builder.add_node("inspect", inspect_code)
builder.add_node("planner", make_plan)
builder.add_node("execute", execute_step)
builder.add_node("verify", verify_code)
builder.add_node("finish", finish)
builder.add_node("stop", stop)

builder.add_edge(START, "inspect")
builder.add_conditional_edges("inspect", after_inspect, ["planner", "finish", "stop"])
builder.add_conditional_edges("planner", after_plan, ["execute", "stop"])
builder.add_edge("execute", "verify")
builder.add_conditional_edges("verify", fixed_after_verify, ["execute", "finish", "stop"])
builder.add_edge("finish", END)
builder.add_edge("stop", END)

fixed_graph = builder.compile(name="fixed_plan_repair")


# 2. Optional run
if RUN_FIXED_PLAN_DEMO:
    reset_workspace()
    fixed_response = fixed_graph.invoke(initial_state(), config={"recursion_limit": 50})
    pprint({key: fixed_response[key] for key in ["status", "error", "plan_history", "steps_used"]})
else:
    print("Fixed-plan demo skipped. Set RUN_FIXED_PLAN_DEMO = True to run it.")

Fixed-plan demo skipped. Set RUN_FIXED_PLAN_DEMO = True to run it.


# 4. Replanning

**Retry** repeats an operation. **Replanning** changes the remaining tasks.
We reuse the planner but provide the latest source, real test failures, recent attempts,
and reflection. `plan` is replaced; `past_steps` and `plan_history` are preserved. [3]

A repair followed by failing tests triggers replanning. A successful inspection step with
remaining work simply continues. Infrastructure failures stop as `blocked` rather than being
misrepresented as evidence that a code patch is wrong.

The next node is connected to the complete graph in Section 5.

In [15]:
# Replanner: same model and schema, different context and instruction.
def replan(state: State):
    try:
        result = request_structured(
            planner,
            "Revise only the remaining plan. Address the observed failures. "
            "Avoid repeating unsuccessful edits and preserve correct existing behavior.\n"
            + task_context(state),
        )
        steps = [step.model_dump() for step in result.steps]
        return {
            "plan": steps,
            "plan_history": [steps],
            "replans": state["replans"] + 1,
        }
    except Exception as error:
        return blocked(error)

# 5. Verification & Reflection

## 5.1. Evidence before self-critique

Verification is already implemented in `verify_source()` and `verify_code()`.
Reflection adds a **brief observation and proposed adjustment**, not a long reasoning transcript.
The reflection may be mistaken; only the independent test report can satisfy `verified()`. [4]

This is reflection within the current task. It is not full Reflexion with persistent cross-trial
memory, and it does not update model weights.

In [16]:
# 1. Reflection schema and agent
class Reflection(BaseModel):
    observation: str = Field(description="One short factual statement supported by the supplied report.")
    adjustment: str = Field(description="One short proposed change for the next attempt.")


reflector = create_agent(
    model=chat,
    tools=[],
    system_prompt=(
        "Review the supplied code and actual test report. Return the Reflection schema. "
        "Keep each field to one sentence. Cite the concrete failing case in the observation. "
        "Propose a targeted adjustment, not a claim that unexecuted tests passed. "
        "Do not invent observations or provide a long reasoning transcript."
    ),
    response_format=ToolStrategy(Reflection),
    middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")],
)


# 2. Reflection node
def reflect(state: State):
    try:
        result = request_structured(reflector, task_context(state))
        return {"reflection": f"Observation: {result.observation}\nAdjustment: {result.adjustment}"}
    except Exception as error:
        return blocked(error)

In [17]:
# 3. Adaptive routing
def adaptive_after_verify(state: State):
    if verified(state):
        return "finish"
    if stop_reason(state):
        return "stop"

    last = state["past_steps"][-1]
    needs_replan = (
        not last["completed"] or last["step"]["kind"] == "repair" or not state["plan"]
    )
    if needs_replan:
        return "reflect" if state["replans"] < MAX_REPLANS else "stop"
    return "execute"


def after_reflect(state: State):
    return "stop" if stop_reason(state) else "replan"

## 5.2. Adaptive graph

```text
START -> inspect -> planner -> execute -> verify
                              ^          |
                              |          +-> finish -> END
                              |          +-> stop -> END
                              +----------+                   (continue the plan)
                              |
                              +-- replan <- reflect <--------+ (revise after failure)
```

The entry router starts at `inspect` for a fresh task, or `verify` when the manual partial-fix
fixture supplies a previous attempt. This is not checkpoint resumption.

In [18]:
# 4. Graph: rebuild rather than attaching a second router to the old builder.
def build_adaptive_graph():
    builder = StateGraph(State)
    builder.add_node("inspect", inspect_code)
    builder.add_node("planner", make_plan)
    builder.add_node("execute", execute_step)
    builder.add_node("verify", verify_code)
    builder.add_node("reflect", reflect)
    builder.add_node("replan", replan)
    builder.add_node("finish", finish)
    builder.add_node("stop", stop)

    builder.add_conditional_edges(
        START, lambda state: "verify" if state["past_steps"] else "inspect",
        ["inspect", "verify"],
    )
    builder.add_conditional_edges("inspect", after_inspect, ["planner", "finish", "stop"])
    builder.add_conditional_edges("planner", after_plan, ["execute", "stop"])
    builder.add_edge("execute", "verify")
    builder.add_conditional_edges(
        "verify", adaptive_after_verify, ["execute", "reflect", "finish", "stop"],
    )
    builder.add_conditional_edges("reflect", after_reflect, ["replan", "stop"])
    builder.add_conditional_edges("replan", after_plan, ["execute", "stop"])
    builder.add_edge("finish", END)
    builder.add_edge("stop", END)
    return builder.compile(name="adaptive_code_repair")


graph = build_adaptive_graph()

## 5.3. Run and inspect

This run begins with the original buggy function. It may succeed on the first repair and never
visit `replan`; that is a valid outcome, not a missing feature. The next subsection deliberately
starts from a manual partial fix to exercise the replanning path.

`stream()` **executes** the graph. Do not call `invoke()` first unless a second execution is intended.
Here `values` exposes each full State without depending on a remote tracing service. [7]

In [19]:
# 5. Run: a single execution, with compact state-change output.
def run_graph(compiled_graph, state: State) -> dict:
    previous = None
    final = state
    for snapshot in compiled_graph.stream(state, config={"recursion_limit": 60}, stream_mode="values"):
        final = snapshot
        report = snapshot.get("test_report", {})
        summary = (
            snapshot["steps_used"], snapshot["replans"],
            len(snapshot["plan"]), report.get("status"), snapshot["status"],
        )
        if summary != previous:
            print(dict(zip(["steps", "replans", "remaining", "tests", "status"], summary)))
            previous = summary
    return final


def show_result(response: dict):
    print("\nStatus:", response["status"])
    print("Reason:", response["error"] or "Current acceptance tests passed.")
    print("\nSource:\n" + response["code"])
    report = response["test_report"]
    print("Tests:", report.get("passed", 0), "/", report.get("total", len(TESTS)))
    for index, plan in enumerate(response["plan_history"]):
        print(f"\nPlan version {index}:")
        pprint(plan)
    if response["reflection"]:
        print("\nLatest reflection:\n" + response["reflection"])


if RUN_ADAPTIVE_DEMO:
    reset_workspace()
    response = run_graph(graph, initial_state())
    show_result(response)
else:
    print("Adaptive demo skipped. Set RUN_ADAPTIVE_DEMO = True to run it.")

{'steps': 0, 'replans': 0, 'remaining': 0, 'tests': None, 'status': 'running'}
{'steps': 0, 'replans': 0, 'remaining': 0, 'tests': 'failed', 'status': 'running'}
{'steps': 0, 'replans': 0, 'remaining': 2, 'tests': 'failed', 'status': 'running'}
{'steps': 1, 'replans': 0, 'remaining': 1, 'tests': None, 'status': 'running'}
{'steps': 1, 'replans': 0, 'remaining': 1, 'tests': 'passed', 'status': 'running'}
{'steps': 1, 'replans': 0, 'remaining': 1, 'tests': 'passed', 'status': 'succeeded'}

Status: succeeded
Reason: Current acceptance tests passed.

Source:
def average(values):
    if len(values) == 0:
        return 0.0
    return sum(values) / len(values)
Tests: 6 / 6

Plan version 0:
[{'done_when': 'The function correctly handles empty lists and returns correct '
               'averages for non-empty lists',
  'kind': 'repair',
  'task': 'Change the function to return 0.0 for empty lists and use '
          'floating-point division'},
 {'done_when': 'All test cases pass without errors

## 5.4. Force a replanning opportunity

This fixture is **manually authored**, not an LLM result. Empty-input handling has already been
added, but fractional results still fail. We start at `verify` with this previous attempt recorded.
Provided the test runner and model are available, the route must enter `reflect -> replan`.
The model still has to produce a real patch; no reference solution is injected into its output.

In [20]:
# 6. Optional controlled demonstration
if RUN_FORCED_REPLAN_DEMO:
    reset_workspace(PARTIAL_SOURCE)
    state = initial_state()
    partial_step = {
        "kind": "repair",
        "task": "Handle empty input only.",
        "done_when": "An empty-input guard is written.",
    }
    state.update(
        steps_used=1,
        past_steps=[{
            "step": partial_step, "completed": True, "changed": True,
            "revision": revision(PARTIAL_SOURCE), "observations": [],
            "assistant_messages": 0, "origin": "manual teaching fixture",
        }],
        plan_history=[[partial_step]],
    )
    replan_graph = build_adaptive_graph()
    replan_response = run_graph(replan_graph, state)
    show_result(replan_response)
else:
    print("Forced replanning skipped. Set RUN_FORCED_REPLAN_DEMO = True to run it.")

Forced replanning skipped. Set RUN_FORCED_REPLAN_DEMO = True to run it.


## 5.5. Deterministic checks — no LLM calls

These assertions test the verifier and controller, not model quality. A reference implementation
is used only to check the test runner. It is not passed to the planner or executor as a solution.
The workspace is restored at the end, so these checks do not overwrite your run's final patch.

In [21]:
# 7. Smoke tests: code failures, current-revision evidence, budgets, and replanning.
REFERENCE_SOURCE = (
    "def average(values):\n"
    "    if not values:\n"
    "        return 0.0\n"
    "    return sum(values) / len(values)\n"
)

saved_source = source_path.read_text(encoding="utf-8")
try:
    assert verify_source(BUGGY_SOURCE)["status"] == "failed"
    assert verify_source(PARTIAL_SOURCE)["status"] == "failed"
    good_report = verify_source(REFERENCE_SOURCE)
    assert good_report["status"] == "passed"
    assert verify_source("import os")["status"] == "failed"

    reset_workspace(REFERENCE_SOURCE)
    state = initial_state()
    state["test_report"] = good_report
    assert verified(state)
    assert finish(state)["status"] == "succeeded"

    reset_workspace(BUGGY_SOURCE)
    assert not verified(state)  # The saved passing report is stale for this file.
    state = initial_state()
    state["steps_used"] = MAX_STEPS
    assert "budget" in stop_reason(state)
    state["steps_used"] = 0
    state["test_report"] = verify_source(BUGGY_SOURCE)
    assert fixed_after_verify({**state, "past_steps": [{"completed": True}]}) == "stop"

    reset_workspace(PARTIAL_SOURCE)
    state = initial_state()
    state["test_report"] = verify_source(PARTIAL_SOURCE)
    state["past_steps"] = [{"step": {"kind": "repair"}, "completed": True}]
    assert adaptive_after_verify(state) == "reflect"
    state["replans"] = MAX_REPLANS
    assert adaptive_after_verify(state) == "stop"
    print("All deterministic smoke tests passed.")
finally:
    reset_workspace(saved_source)

All deterministic smoke tests passed.


## 5.6. Optional visualization and traces

The graph drawing shows the outer workflow. Tool calls inside the executor are recorded in
`past_steps[*]["observations"]`. The default drawing below is Mermaid text and works offline.
The optional PNG renderer may contact an external rendering service. [8]

In [22]:
# 8. Visualize the graph structure without executing it.
print(graph.get_graph().draw_mermaid())

# Optional PNG rendering:
# Path("diagrams").mkdir(exist_ok=True)
# display(Image(graph.get_graph().draw_mermaid_png(
#     output_file_path="diagrams/4_reasoning_planning.png",
# )))

# Inspect tool observations from the last adaptive run:
# for attempt in response["past_steps"]:
#     print(attempt["step"]["task"])
#     pprint(attempt["observations"])

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	inspect(inspect)
	planner(planner)
	execute(execute)
	verify(verify)
	reflect(reflect)
	replan(replan)
	finish(finish)
	stop(stop)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> inspect;
	__start__ -.-> verify;
	execute --> verify;
	inspect -.-> finish;
	inspect -.-> planner;
	inspect -.-> stop;
	planner -.-> execute;
	planner -.-> stop;
	reflect -.-> replan;
	reflect -.-> stop;
	replan -.-> execute;
	replan -.-> stop;
	verify -.-> execute;
	verify -.-> finish;
	verify -.-> reflect;
	verify -.-> stop;
	finish --> __end__;
	stop --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



# Summary

| Pattern | What changes? | Who decides success? |
|---|---|---|
| ReAct-style agent | Next tool call after each observation | External verifier |
| Fixed plan | Next task from the initial plan | External verifier |
| Adaptive plan | Remaining tasks after failure or new evidence | External verifier |
| Reflection | Brief feedback for the next planning attempt | Not the reflector |

**Plan completion is not task success.** A passing report must match the current source revision.

| Limit | Scope |
|---|---|
| `MAX_MODEL_CALLS` | Each executor invocation; planner/reflector have their own limit of 3. |
| `MAX_STEPS` | Total executor invocations in one outer graph run. |
| `MAX_REPLANS` | Number of successful plan revisions. |
| Two unchanged repairs | Stop repeatedly proposing no code change. |
| Test timeout | Bound each test subprocess. |
| Model-client timeout | Bound an HTTP request; not a strict whole-task deadline. |

This is one sequential teaching workspace. Do not run cells concurrently. Workspace edits are
side effects; ordinary graph State alone would not roll them back or restore them after a crash.
There is no implicit agent training, multi-agent coordination, or production deployment here.

Tree of Thoughts / multiple-patch search is an optional extension: keep separate candidate
workspaces, test each candidate, and compare evidence. It is deliberately not implemented here.
A planner is not automatically better than a small reactive agent on a task this simple.

<details>
<summary>Note) Troubleshooting</summary><br>

| Symptom | Check |
|---|---|
| Connection error | Start Ollama and confirm the configured model exists. |
| No `structured_response` | The model did not finish schema output within its call budget. Inspect the model behavior before raising limits. |
| `apply_patch` rejection | Supply only the complete supported function; no Markdown fences or extra syntax. |
| `blocked` | Read `error`; it is an infrastructure/model-output problem, not a successful repair. |
| `incomplete` | Read the test report and attempts; a plan or budget ended before acceptance. |
| No replanning happened | The first repair may have passed. Enable the manual partial-fix demo. |

The notebook never silently substitutes the reference solution when the model fails.

</details>

## Validation note

At creation time, Python syntax, the actual subprocess test runner, tool implementations,
and controller/node functions were checked with deterministic fixtures (41 additional assertions).
**LangGraph runtime execution
and real Ollama inference were not run in the authoring environment.** The notebook contains no
fabricated model outputs. Run locally to inspect your own model's plans and tool calls.

## References

[1] [LangChain: ChatOllama integration](https://docs.langchain.com/oss/python/integrations/chat/ollama)  
[2] [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)  
[3] [LangChain: Plan-and-Execute Agents](https://www.langchain.com/blog/planning-agents)  
[4] [Reflexion: Language Agents with Verbal Reinforcement Learning](https://arxiv.org/abs/2303.11366)  
[5] [Python: subprocess](https://docs.python.org/3/library/subprocess.html)  
[6] [LangChain: Structured output](https://docs.langchain.com/oss/python/langchain/structured-output)  
[7] [LangGraph: Streaming](https://docs.langchain.com/oss/python/langgraph/streaming)  
[8] [LangGraph: Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)  
[9] [LangChain: ModelCallLimitMiddleware](https://reference.langchain.com/python/langchain/agents/middleware/model_call_limit/ModelCallLimitMiddleware)

API references describe library behavior. The task, checks, budgets, and controller design are
teaching choices for this notebook, not claims that the cited papers prescribe this exact code.